# 05 · Orquestación y ablación: ¿cuánto aporta cada modelo y cada modalidad?

Es el argumento central del proyecto: **combinar modelos y modalidades supera a cualquiera por separado**.
Lo comprobamos con un estudio de ablación sobre casos multimodales y ajustamos los pesos de la fusión.

**Casos.** Convertimos los 61 mensajes del test escrito a mano (notebook 01) en casos **multimodales
reales**:

* los SMS, WhatsApp, correos y webs se **renderizan como capturas** (el sistema solo ve la imagen y tiene
  que leerla con Qwen2.5-VL);
* las transcripciones de llamadas se **locutan** con MMS-TTS y pasan por el canal telefónico simulado (el
  sistema solo ve el audio y tiene que transcribirlo con Whisper).

Los metadatos (remitente, hora) son **idénticos** para estafas y legítimos, para que el remitente no
delate la etiqueta. Ninguno de estos textos se usó para entrenar.

**Señales** que recoge el orquestador (`src/diputado/core/pipeline.py`) para cada caso:

| Señal | Modelo | Modalidad de origen |
|---|---|---|
| `tacticnet` | e5 + TacticNet (Keras) | Texto (leído de la imagen o transcrito) |
| `llm` | Qwen3 8B | Texto + contexto visual |
| `reglas` | Regex + rapidfuzz | Texto + enlaces leídos por el VLM |
| `campana` | e5 + SigLIP2 | Texto **e imagen** |
| `voz`, `acustica` | CLAP + VozSinteticaNet (Keras) | Audio |

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from diputado import config

config.ensure_dirs()
config.ensure_ffmpeg_on_path()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Markdown, display
from scipy.optimize import minimize
from sklearn.metrics import brier_score_loss, confusion_matrix, f1_score, roc_auc_score

from diputado.ai import ollama_server
from diputado.core import fusion, pipeline
from diputado.core.schemas import CaseInput
from diputado.media import render

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
ollama_server.ensure_server(install_if_missing=False)
pipeline._store = lambda r: None  # no mezclamos estos experimentos con el histórico de la consola

## 1. Construcción de los casos multimodales

In [ ]:
from diputado.ai import tts
from diputado.ai.audio_io import resample
from diputado.data.telephony import phone_channel

test = pd.read_csv(config.DATA_DIR / "test_manual.csv")
case_dir = config.SYNTHETIC_DIR / "ablacion"
case_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
cases = []
for i, r in test.iterrows():
    if r["canal"] == "llamada":
        p = case_dir / f"caso_{i:02d}.wav"
        if not p.exists():
            audio, sr = tts.synthesize(r["texto"], seed=i)
            sf.write(p, phone_channel(resample(audio, sr, 16000), 16000, rng), 16000)
        cases.append({"i": i, "audio": str(p), "imagen": None})
    else:
        p = case_dir / f"caso_{i:02d}.png"
        if not p.exists():
            if r["canal"] == "whatsapp":
                img = render.render_whatsapp("Contacto", r["texto"], hour="12:00", unknown=False)
            elif r["canal"] == "email":
                img = render.render_email("notificaciones@correo.es", "Aviso", r["texto"], hour="12:00")
            elif r["canal"] == "web":
                img = render.render_web("Página web", r["texto"])
            else:
                img = render.render_sms("Aviso", r["texto"], hour="12:00")
            render.save(img, p)
        cases.append({"i": i, "audio": None, "imagen": str(p)})
print(f"{len(cases)} casos · {sum(c['audio'] is not None for c in cases)} llamadas · {sum(c['imagen'] is not None for c in cases)} capturas")

## 2. Ejecución del orquestador completo sobre cada caso

Guardamos en caché (`artifacts/ablacion_senales.csv`) las señales de cada caso para que el resto del
notebook (y quien lo reproduzca) no tenga que repetir los ~15 minutos de inferencia.

In [ ]:
CACHE = config.ARTIFACTS_DIR / "ablacion_senales.csv"
done = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame()
rows = done.to_dict("records")
seen = set(done["i"]) if len(done) else set()
t0 = time.perf_counter()
for c in cases:
    if c["i"] in seen:
        continue
    r = pipeline.run(CaseInput(image=c["imagen"], audio=c["audio"], source="ablacion"))
    camp = r.campaigns[0] if r.campaigns else {}
    rows.append({
        "i": c["i"], "modalidad": "audio" if c["audio"] else "imagen",
        "tacticnet": (r.tactic_model or {}).get("scam_prob"), "llm": (r.llm or {}).get("probabilidad_estafa"),
        "reglas": fusion.rules_to_prob(r.extract["red_flag_score"]) if r.extract else None,
        "campana": fusion.campaign_to_prob(camp["similitud"]) if camp else None,
        "campana_texto": camp.get("detalle", {}).get("texto"), "campana_imagen": camp.get("detalle", {}).get("imagen"),
        "voz": (r.voice or {}).get("synthetic_prob"),
        "acustica": (r.acoustic or {}).get("Voz robótica o sintética", 0) + (r.acoustic or {}).get("Menú automático (IVR)", 0) if r.acoustic else None,
        "regla_dura": bool(r.extract and r.extract["hard_rule"]),
        "riesgo_app": r.risk, "nivel_app": r.level, "ms": r.total_ms(), "errores": "; ".join(r.errors),
        "texto_leido": r.evidence_text[:300],
    })
    pd.DataFrame(rows).to_csv(CACHE, index=False)
print(f"Inferencia: {time.perf_counter() - t0:.0f} s")
S = pd.DataFrame(rows).sort_values("i").reset_index(drop=True)
S["estafa"] = test.loc[S["i"], "estafa"].values
S["canal"] = test.loc[S["i"], "canal"].values
print("Casos con errores:", (S["errores"].fillna("") != "").sum())
display(S.head())

## 3. Cada señal por separado

In [ ]:
SIGNALS = ["tacticnet", "llm", "reglas", "campana", "voz", "acustica"]
y = S["estafa"].values


def metrics(p, y_true=None, mask=None):
    y_true = y if y_true is None else y_true
    if mask is not None:
        p, y_true = p[mask], y_true[mask]
    ok = ~np.isnan(p)
    p, y_true = p[ok], y_true[ok]
    if len(np.unique(y_true)) < 2:
        return {"n": len(p), "AUC": np.nan, "F1": np.nan, "Brier": np.nan}
    return {"n": len(p), "AUC": roc_auc_score(y_true, p), "F1": f1_score(y_true, p >= 0.5), "Brier": brier_score_loss(y_true, np.clip(p, 0, 1))}


single = pd.DataFrame({s: metrics(S[s].astype(float).values) for s in SIGNALS}).T
single.index = [fusion.SIGNAL_LABELS[s] for s in single.index]
display(single.round(3))

## 4. Fusión: pesos iniciales frente a pesos ajustados

El riesgo es `sigmoide(b + Σ w_i · logit(p_i))`. Ajustamos `b` y `w` minimizando la log-loss con
regularización L2 y **pesos no negativos** (una señal de fraude nunca debe restar riesgo por ser alta).
Las señales de audio son **amplificadoras**: antes del logit las recortamos a ≥ 0,5, igual que en la
aplicación. Para no evaluarnos con los mismos datos con los que ajustamos, usamos **validación cruzada
dejando uno fuera** (LOO): cada caso se predice con pesos ajustados sin él.

In [ ]:
def features(df, names):
    F = np.zeros((len(df), len(names)))
    for j, n in enumerate(names):
        p = df[n].astype(float).values
        if n in fusion.AMPLIFIERS:
            p = np.where(np.isnan(p), np.nan, np.maximum(p, 0.5))
        F[:, j] = np.where(np.isnan(p), 0.0, [fusion.logit(v) if not np.isnan(v) else 0 for v in p])
    return F


def fit(F, yy, l2=0.5):
    def loss(theta):
        z = theta[0] + F @ theta[1:]
        p = 1 / (1 + np.exp(-z))
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return -np.mean(yy * np.log(p) + (1 - yy) * np.log(1 - p)) + l2 * np.sum(theta[1:] ** 2) / len(yy)
    x0 = np.r_[0.0, np.full(F.shape[1], 0.5)]
    res = minimize(loss, x0, bounds=[(-3, 3)] + [(0, 3)] * F.shape[1], method="L-BFGS-B")
    return res.x


def predict(theta, F):
    return 1 / (1 + np.exp(-(theta[0] + F @ theta[1:])))


def loo(names, l2=0.5):
    F = features(S, names)
    out = np.zeros(len(S))
    for k in range(len(S)):
        m = np.arange(len(S)) != k
        out[k] = predict(fit(F[m], y[m], l2), F[k:k + 1])[0]
    return out


def default_fusion(names, hard=True):
    return np.array([fusion.fuse({n: (None if pd.isna(row[n]) else float(row[n])) for n in names},
                                 hard_rule=hard and row["regla_dura"], params=fusion.DEFAULT).risk for _, row in S.iterrows()])


combos = {
    "Solo TacticNet": ["tacticnet"],
    "Solo Qwen3": ["llm"],
    "Solo reglas": ["reglas"],
    "Texto: TacticNet + Qwen3": ["tacticnet", "llm"],
    "Texto: TacticNet + Qwen3 + reglas": ["tacticnet", "llm", "reglas"],
    "Texto + campañas (texto e imagen)": ["tacticnet", "llm", "reglas", "campana"],
    "Todo (texto + imagen + audio)": SIGNALS,
}
abl = []
for name, names in combos.items():
    abl.append({"configuración": name, "pesos": "iniciales", **metrics(default_fusion(names, hard=False))})
    abl.append({"configuración": name, "pesos": "ajustados (LOO)", **metrics(loo(names))})
abl.append({"configuración": "Todo + regla dura (aplicación)", "pesos": "iniciales", **metrics(default_fusion(SIGNALS))})
abl = pd.DataFrame(abl)
display(abl.pivot(index="configuración", columns="pesos", values=["AUC", "F1", "Brier"]).reindex(list(combos) + ["Todo + regla dura (aplicación)"]).round(3))

In [ ]:
plot = abl[abl["pesos"] == "ajustados (LOO)"].set_index("configuración")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot["AUC"].plot.barh(ax=axes[0], color=["#9aa5b8"] * 3 + ["#e89128"] * 3 + ["#d7263d"])
axes[0].set_xlim(0.5, 1); axes[0].set_title("ROC-AUC (LOO)")
plot["Brier"].plot.barh(ax=axes[1], color=["#9aa5b8"] * 3 + ["#e89128"] * 3 + ["#d7263d"])
axes[1].set_title("Brier (menor es mejor)"); axes[1].set_yticklabels([])
plt.tight_layout()

### ¿Es significativa la mejora? Bootstrap pareado

Con 61 casos las diferencias pueden ser ruido. Remuestreamos los casos 2.000 veces y calculamos el
intervalo de confianza del 95 % de la diferencia de Brier entre el sistema completo y el mejor modelo
individual.

In [ ]:
p_full = loo(SIGNALS)
best_single = min(["tacticnet", "llm", "reglas"], key=lambda s: brier_score_loss(y, np.nan_to_num(S[s].astype(float).values, nan=0.5)))
p_best = np.nan_to_num(S[best_single].astype(float).values, nan=0.5)
rng = np.random.default_rng(0)
diffs, auc_d = [], []
for _ in range(2000):
    b = rng.integers(0, len(y), len(y))
    if len(np.unique(y[b])) < 2:
        continue
    diffs.append(brier_score_loss(y[b], p_best[b]) - brier_score_loss(y[b], p_full[b]))
    auc_d.append(roc_auc_score(y[b], p_full[b]) - roc_auc_score(y[b], p_best[b]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
alo, ahi = np.percentile(auc_d, [2.5, 97.5])
display(Markdown(f"Mejor señal individual: **{fusion.SIGNAL_LABELS[best_single]}**.  \n"
                 f"Mejora de Brier del sistema fusionado: **{np.mean(diffs):+.3f}** (IC 95 %: {lo:+.3f} a {hi:+.3f}).  \n"
                 f"Mejora de AUC: **{np.mean(auc_d):+.3f}** (IC 95 %: {alo:+.3f} a {ahi:+.3f})."))

## 5. Pesos finales y exportación a `artifacts/fusion.json`

Ajustamos con todos los casos. Para no sobreajustar a 61 casos, **mezclamos** los pesos ajustados con
los iniciales razonados (media 50/50): es una forma simple de regularización hacia un *prior* experto.

In [ ]:
F_all = features(S, SIGNALS)
theta = fit(F_all, y)
fitted = dict(zip(SIGNALS, theta[1:]))
blend = {s: round(0.5 * fitted[s] + 0.5 * fusion.DEFAULT["weights"][s], 3) for s in SIGNALS}
bias = round(0.5 * theta[0] + 0.5 * fusion.DEFAULT["bias"], 3)
wt = pd.DataFrame({"iniciales": fusion.DEFAULT["weights"], "ajustados": {k: round(v, 3) for k, v in fitted.items()}, "finales (mezcla)": blend})
wt.index = [fusion.SIGNAL_LABELS[s] for s in wt.index]
display(wt)
print(f"Sesgo: inicial {fusion.DEFAULT['bias']} · ajustado {theta[0]:.3f} · final {bias}")

params = {"bias": bias, "weights": blend, "source": "notebook 05: mezcla 50/50 de pesos ajustados (61 casos multimodales) y pesos iniciales"}
p_final = np.array([fusion.fuse({n: (None if pd.isna(row[n]) else float(row[n])) for n in SIGNALS}, hard_rule=row["regla_dura"], params=params).risk
                    for _, row in S.iterrows()])
print("Con los pesos finales:", {k: round(v, 3) for k, v in metrics(p_final).items()})
(config.ARTIFACTS_DIR / "fusion.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")

## 6. El semáforo: tres niveles

La aplicación no da una probabilidad, da un **semáforo**: verde (< 0,35), ámbar (0,35-0,65) y rojo (≥ 0,65).
El ámbar es deliberado: es mejor decir «compruébalo» que equivocarse con seguridad.

In [ ]:
levels = np.array([fusion.level_for(p) for p in p_final])
tab = pd.crosstab(pd.Series(np.where(y == 1, "estafa", "legítimo"), name="real"), pd.Series(levels, name="semáforo"))
display(tab.reindex(columns=["verde", "ambar", "rojo"], fill_value=0))
fn = ((y == 1) & (levels == "verde")).sum()
fp = ((y == 0) & (levels == "rojo")).sum()
display(Markdown(f"**Errores graves:** {fn} estafas en verde y {fp} legítimos en rojo, de {len(y)} casos."))

In [ ]:
S["riesgo_final"] = p_final
S["nivel_final"] = levels
bad = S[((S["estafa"] == 1) & (S["nivel_final"] != "rojo")) | ((S["estafa"] == 0) & (S["nivel_final"] != "verde"))]
display(bad[["i", "canal", "estafa", "nivel_final", "riesgo_final", "tacticnet", "llm", "reglas", "texto_leido"]].round(2))

## 7. Por modalidad de entrada

In [ ]:
by_mod = pd.DataFrame({m: metrics(p_final, mask=(S["modalidad"] == m).values) for m in ("imagen", "audio")}).T
display(by_mod.round(3))
print(f"Latencia media del análisis: imagen {S.loc[S.modalidad == 'imagen', 'ms'].mean() / 1000:.1f} s · "
      f"audio {S.loc[S.modalidad == 'audio', 'ms'].mean() / 1000:.1f} s")

In [ ]:
summary = {"casos": len(S), "senales_individuales": single.round(4).to_dict(orient="index"),
           "ablacion": abl.round(4).to_dict(orient="records"),
           "bootstrap": {"mejor_individual": best_single, "mejora_brier": [round(float(np.mean(diffs)), 4), round(float(lo), 4), round(float(hi), 4)],
                         "mejora_auc": [round(float(np.mean(auc_d)), 4), round(float(alo), 4), round(float(ahi), 4)]},
           "semaforo": tab.to_dict(), "final": {k: round(float(v), 4) for k, v in metrics(p_final).items()}}
(config.ARTIFACTS_DIR / "ablacion.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

## Conclusiones

* Ninguna señal sola es suficiente: las reglas son precisas pero cortas de cobertura, TacticNet es rápido
  pero aprendió de datos sintéticos, y el LLM generaliza pero se equivoca con seguridad en algunos casos.
* La fusión combina sus fortalezas, y la **regla dura** (pedir códigos o instalar apps de control remoto)
  cubre el caso más dañino con una garantía determinista.
* Las señales de audio aportan poco en este conjunto porque las llamadas legítimas y las fraudulentas usan
  la misma voz sintética (MMS): es exactamente la razón por la que en la aplicación son amplificadoras y
  no decisivas.
* Limitación: 61 casos, construidos a partir de nuestro propio test. Los intervalos de confianza lo
  reflejan; en producción el banco ajustaría los pesos con sus casos reales en minutos.